# Redefining Epidemiological Waves: \\ Structural Stability and Global Empirical Validation in Sobolev Spaces

**Author:** Santi García-Cremades (Miguel Hernandez University of Elche)  
**Paper:** *Topological Condemnation: Structural Stability and Kinematic Detection of Epidemiological Waves in Sobolev Spaces*

### Overview
This notebook provides the complete, reproducible pipeline for detecting structural epidemic waves from highly noisy institutional data. By shifting the paradigm from absolute infection counts to the geometric duration of structural instability—a metric we term **Topological Condemnation**—this framework offers a domain-agnostic approach to wave detection.

In [2]:
import os
import numpy as np
import pandas as pd
import requests
import scipy.sparse as sparse
from scipy.sparse.linalg import spsolve
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

# Configuration and Hyperparameters
LAMBDA_H3 = 5000 
MIN_AMPLITUDE_IA14 = 50  
MIN_POPULATION = 100000  
FIG_DIR = "figures"

if not os.path.exists(FIG_DIR):
    os.makedirs(FIG_DIR)

## 1. Core Mathematical Framework

Raw epidemiological data is fundamentally non-stationary and corrupted by institutional artifacts (e.g., weekend under-reporting). Direct differentiation of such data amplifies high-frequency noise quadratically, making peak detection structurally unstable.

To restore geometric stability, we project the raw incidence series $Y$ into the Sobolev space $H^3$ by solving the following discrete Tikhonov regularization problem:

$$\hat{x} = \arg\min_{x \in \mathbb{R}^n} \|x - Y\|_2^2 + \lambda \|\nabla^3 x\|_2^2$$

Once the continuous state $\hat{x}$ is recovered, epidemic waves are defined topologically through the zero-crossings of the kinematic derivatives (velocity $v_t$ and acceleration $a_t$).

In [3]:
def tikhonov_h3_sparse(y, lam=5000):
    """
    Projects the raw time series into the Sobolev space H^3
    using sparse matrix operations for O(n) computational efficiency.
    """
    n = len(y)
    if n < 4: return y
    D3 = sparse.diags([-1, 3, -3, 1], [0, 1, 2, 3], shape=(n-3, n))
    I = sparse.eye(n)
    A = I + lam * D3.T.dot(D3)
    return spsolve(A, y)

def find_kinematic_waves(x_h3, min_amplitude=50):
    """
    Identifies structurally stable epidemic waves based on the 
    zero-crossings of velocity and acceleration in H^3 space.
    """
    v = np.gradient(x_h3)
    a = np.gradient(v)
    waves = []
    in_wave = False
    start_idx = 0
    
    for i in range(1, len(v) - 1):
        if v[i-1] <= 0 and v[i] > 0 and not in_wave:
            in_wave = True
            start_idx = i
        elif v[i-1] > 0 and v[i] <= 0 and a[i] < 0 and in_wave:
            peak_idx = i
            end_idx = peak_idx
            while end_idx < len(v) - 1 and v[end_idx] <= 0:
                end_idx += 1
            if (x_h3[peak_idx] - x_h3[start_idx]) >= min_amplitude:
                waves.append({'start_idx': start_idx, 'peak_idx': peak_idx, 'end_idx': end_idx})
            in_wave = False
    return waves

## 2. Global Data Acquisition

To empirically validate our framework, we fetch global daily COVID-19 incidence data from the **Johns Hopkins University (JHU)** repository. We cross-reference this with demographic data from the **World Bank API** to normalize the incidence into standard rates (IA14: 14-day cumulative incidence per 100,000 inhabitants).

In [4]:
print("Fetching global COVID-19 time series from JHU...")
url_jhu = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_confirmed_global.csv"
df_jhu = pd.read_csv(url_jhu)
df_jhu = df_jhu.drop(columns=['Lat', 'Long', 'Province/State']).groupby('Country/Region').sum().T
df_jhu.index = pd.to_datetime(df_jhu.index)

print("Fetching population metrics from the World Bank API...")
url_pop = "https://api.worldbank.org/v2/country/all/indicator/SP.POP.TOTL?format=json&per_page=300&date=2021"
response = requests.get(url_pop).json()
pop_data = {item['country']['value']: item['value'] for item in response[1] if item['value'] is not None}

pop_map = {
    'US': 'United States', 'Russia': 'Russian Federation', 
    'Korea, South': 'Korea, Rep.', 'Iran': 'Iran, Islamic Rep.',
    'Egypt': 'Egypt, Arab Rep.', 'Venezuela': 'Venezuela, RB',
    'Czechia': 'Czech Republic', 'Slovakia': 'Slovak Republic',
    'Syria': 'Syrian Arab Republic', 'Yemen': 'Yemen, Rep.'
}

Fetching global COVID-19 time series from JHU...
Fetching population metrics from the World Bank API...


## 3. Global Execution & Topo-Kinematic Analysis

In this section, we iterate over 170+ nations. For each country, we:
1. Compute the normalized daily incidence.
2. Apply the $H^3$ Sobolev topological filter.
3. Extract all structural epidemic waves.
4. Calculate the total **Topological Condemnation** (cumulative days spent in an epidemic state).
5. Generate and save a visualization plot, highlighting the growth phase (red) and decay phase (green).

In [5]:
all_waves_detailed = []
summary_data = []
countries_plotted = 0

print("Processing kinematics and rendering global plots...")

for country in df_jhu.columns:
    wb_name = pop_map.get(country, country)
    pop = pop_data.get(wb_name, None)
    
    if pop is None or pop < MIN_POPULATION: continue
        
    cumulative = df_jhu[country].values
    incidence = np.diff(cumulative, prepend=0)
    incidence[incidence < 0] = 0  
    
    df_country = pd.DataFrame({'Date': df_jhu.index, 'Incidence': incidence})
    df_country['IA14'] = df_country['Incidence'].rolling(window=14, min_periods=1).sum() / (pop / 100000)
    
    ia14_raw = df_country['IA14'].values
    ia14_h3 = tikhonov_h3_sparse(ia14_raw, LAMBDA_H3)
    
    waves = find_kinematic_waves(ia14_h3, MIN_AMPLITUDE_IA14)
    if not waves: continue
        
    total_epidemic_days = 0
    total_infected_in_waves = 0
    
    # ---------------- PLOTTING SETUP ----------------
    plt.figure(figsize=(12, 6))
    plt.plot(df_country['Date'], ia14_raw, color='lightgray', alpha=0.6, label='Raw IA14')
    plt.plot(df_country['Date'], ia14_h3, color='black', linewidth=1.5, label='H3 Regularized IA14')
    
    for w_idx, w in enumerate(waves):
        start_idx, peak_idx, end_idx = w['start_idx'], w['peak_idx'], w['end_idx']
        start, peak, end = df_country['Date'].iloc[start_idx], df_country['Date'].iloc[peak_idx], df_country['Date'].iloc[end_idx]
        
        duration = end_idx - start_idx
        infected = incidence[start_idx:end_idx].sum()
        
        total_epidemic_days += duration
        total_infected_in_waves += infected
        
        all_waves_detailed.append({
            'Country': country, 'Wave_Number': w_idx + 1,
            'Start_Date': start.strftime('%Y-%m-%d'), 'Peak_Date': peak.strftime('%Y-%m-%d'), 'End_Date': end.strftime('%Y-%m-%d'),
            'Duration_Days': duration, 'Peak_IA14': round(ia14_h3[peak_idx], 2), 'Infected_In_Wave': int(infected)
        })
        
        # Draw the Red Area (Growth phase: Start to Peak)
        plt.fill_between(df_country['Date'].iloc[start_idx:peak_idx+1], 
                         ia14_h3[start_idx:peak_idx+1], 
                         color='red', alpha=0.3)
                         
        # Draw the Green Area (Decay phase: Peak to End)
        plt.fill_between(df_country['Date'].iloc[peak_idx:end_idx+1], 
                         ia14_h3[peak_idx:end_idx+1], 
                         color='green', alpha=0.3)
                         
        # Draw Peak Line
        plt.axvline(peak, color='black', linestyle='--', alpha=0.5)

    summary_data.append({
        'Country': country, 'Total_Waves': len(waves),
        'Total_Infected': int(total_infected_in_waves), 'Total_Epidemic_Days': total_epidemic_days
    })
    
    # ---------------- FINALIZE PLOT ----------------
    plt.title(f"{country} - Topo-Kinematic Epidemic Waves ($H^3$ Space)", fontsize=14)
    plt.ylabel("Incidence per 100k (IA14)")
    
    # Custom legend to explain colors
    from matplotlib.patches import Patch
    handles, labels = plt.gca().get_legend_handles_labels()
    handles.extend([Patch(facecolor='red', alpha=0.3), Patch(facecolor='green', alpha=0.3)])
    labels.extend(['Growth Phase (Red)', 'Decay Phase (Green)'])
    plt.legend(handles, labels)
    
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/{country.replace('*', '').replace(' ', '_')}_H3.png", dpi=150)
    plt.close()
    
    countries_plotted += 1

print(f"Successfully processed {countries_plotted} nations.")

Processing kinematics and rendering global plots...
Successfully processed 135 nations.


## 4. Result Export

Finally, we export the calculated topological boundaries and the global condemnation summaries into CSV format for further statistical analysis.

In [6]:
df_detailed = pd.DataFrame(all_waves_detailed)
df_detailed.to_csv("Detailed_Waves_H3.csv", index=False)

df_summary = pd.DataFrame(summary_data).sort_values(by='Total_Epidemic_Days', ascending=False).reset_index(drop=True)
df_summary.to_csv("Country_Summary_H3.csv", index=False)

print("Data exported successfully to CSV.")
df_summary.head(10)

Data exported successfully to CSV.


,Country,Total_Waves,Total_Infected,Total_Epidemic_Days
0,Argentina,8,10044945,1094
1,Serbia,9,2460450,1071
2,Italy,9,24989989,1066
3,Luxembourg,15,315985,1059
4,France,10,40216813,1048
5,US,10,100865597,1028
6,United Kingdom,11,24235692,1025
7,Bolivia,6,1185607,1023
8,Germany,12,38176054,1012
9,Austria,11,5906830,1008


In [7]:
import shutil

print("Compressing the 'figures' folder into a .zip file...")

# Esto crea un archivo llamado 'Epidemic_Waves_Figures.zip' con el contenido de la carpeta 'figures'
shutil.make_archive('Epidemic_Waves_Figures', 'zip', FIG_DIR)

print("Compression complete! You can now download 'Epidemic_Waves_Figures.zip'.")

Compressing the 'figures' folder into a .zip file...
Compression complete! You can now download 'Epidemic_Waves_Figures.zip'.
